# Dairy Milk Production Data Inspection

This notebook checks the CSV structure, data quality, missingness, production distributions, and cow-level coverage before missing-value modeling.

In [ ]:
from pathlib import Path
import csv
import matplotlib.pyplot as plt
import pandas as pd

DATA_PATH = Path('project_1_ANSC_4040_dataset.csv')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda value: f'{value:,.3f}')
plt.style.use('seaborn-v0_8-whitegrid')

## Load and validate the CSV

The source header contains a missing comma between `Flow30_60` and `Session`. The loader repairs that label and verifies that every row has 11 fields.

In [ ]:
with DATA_PATH.open(newline='', encoding='utf-8-sig') as file:
    rows = list(csv.reader(file))

raw_header = rows[0]
expected_columns = ['AnimalNumber', 'LactationNumber', 'DaysInMilk', 'ReproductionStatus', 'Avgmilkflow', 'Flow30_60', 'Session', 'YieldFirst2Min_Session', 'YieldSession', 'DurationSession_sec', 'milking']
data_rows = rows[1:]
field_counts = pd.Series([len(row) for row in data_rows]).value_counts().sort_index()
print('Raw header:', raw_header)
print('Field counts by row:')
display(field_counts.rename('number_of_rows').to_frame())
if any(len(row) != len(expected_columns) for row in data_rows):
    raise ValueError('At least one data row has an unexpected number of fields.')
df = pd.DataFrame(data_rows, columns=expected_columns)
print(f'Validated shape: {df.shape}')
display(df.head())

In [ ]:
numeric_columns = ['AnimalNumber', 'LactationNumber', 'DaysInMilk', 'Avgmilkflow', 'Flow30_60', 'Session', 'YieldFirst2Min_Session', 'YieldSession', 'DurationSession_sec', 'milking']
df[numeric_columns] = df[numeric_columns].apply(pd.to_numeric, errors='coerce')
df['ReproductionStatus'] = df['ReproductionStatus'].astype('category')
df.info()

## Data quality and missingness

In [ ]:
overview = pd.DataFrame({'metric': ['Rows', 'Columns', 'Unique animals', 'Duplicate rows'], 'value': [len(df), df.shape[1], df['AnimalNumber'].nunique(), int(df.duplicated().sum())]})
display(overview)
missing = (df.isna().sum().rename('missing_count').to_frame().assign(missing_percent=lambda table: 100 * table['missing_count'] / len(df)).query('missing_count > 0').sort_values('missing_count', ascending=False))
display(missing if not missing.empty else pd.DataFrame({'result': ['No missing values detected']}))
display(df['ReproductionStatus'].value_counts(dropna=False).rename('count').to_frame())
display(df.select_dtypes(include='number').describe().T)

## Production distributions and relationships

In [ ]:
plot_columns = ['YieldSession', 'YieldFirst2Min_Session', 'Avgmilkflow', 'Flow30_60', 'DurationSession_sec', 'DaysInMilk']
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for axis, column in zip(axes.flat, plot_columns):
    df[column].plot(kind='hist', bins=30, ax=axis, color='#2f6f73', edgecolor='white')
    axis.set_title(column)
    axis.set_xlabel('')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df.boxplot(column='YieldSession', by='ReproductionStatus', ax=axes[0], grid=False)
axes[0].set_title('Session Yield by Reproduction Status')
axes[0].set_xlabel('Reproduction status')
axes[1].scatter(df['DaysInMilk'], df['YieldSession'], alpha=0.15, s=10, color='#d05a3b')
axes[1].set_title('Session Yield vs. Days in Milk')
axes[1].set_xlabel('Days in milk')
axes[1].set_ylabel('Session yield')
plt.suptitle('')
plt.tight_layout()
plt.show()
display(df[['Avgmilkflow', 'Flow30_60', 'YieldFirst2Min_Session', 'YieldSession', 'DurationSession_sec', 'DaysInMilk']].corr())

## Cow-level coverage

In [ ]:
animal_summary = (df.groupby('AnimalNumber', observed=True).agg(records=('AnimalNumber', 'size'), mean_yield=('YieldSession', 'mean'), yield_sd=('YieldSession', 'std'), min_days_in_milk=('DaysInMilk', 'min'), max_days_in_milk=('DaysInMilk', 'max')).sort_values('records', ascending=False))
display(animal_summary.head(10))
display(animal_summary[['records', 'mean_yield', 'yield_sd']].describe())
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
animal_summary['records'].plot.hist(bins=30, ax=axes[0], color='#2f6f73', edgecolor='white')
axes[0].set_title('Records per Animal')
animal_summary['mean_yield'].plot.hist(bins=30, ax=axes[1], color='#d05a3b', edgecolor='white')
axes[1].set_title('Mean Session Yield per Animal')
plt.tight_layout()
plt.show()

## Next modeling checks

Use these outputs to define the missingness pattern and identify repeated observations for each animal. Then create artificial gaps in known `YieldSession` values and compare cow-level averages with interpolation using mean absolute error (MAE).